In [1]:
# ====================================================
# 📦 IMPORT LIBRARIES FOR RECOMMENDATION ENGINE
# ====================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
import warnings

# ML Libraries
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')

print("✅ All libraries imported!")
print("🧠 Ready to build Recommendation Engine!")

✅ All libraries imported!
🧠 Ready to build Recommendation Engine!


In [2]:
# ====================================================
# 📂 LOAD DATA AND TRAINED MODELS
# ====================================================

PROCESSED_PATH = Path("../dataset/processed/")
MODELS_PATH = Path("../backend/ml/models/")

# Load cleaned data
cards_df = pd.read_csv(PROCESSED_PATH / "cards_cleaned.csv")
decks_df = pd.read_csv(PROCESSED_PATH / "decks_cleaned.csv")

# Load trained models
scaler = joblib.load(MODELS_PATH / "scaler.pkl")
win_rate_model = joblib.load(MODELS_PATH / "win_rate_predictor.pkl")

# Card lookup dictionary
card_lookup = cards_df.set_index('name').to_dict('index')

print("✅ Data and models loaded!")
print(f"📦 Cards:  {cards_df.shape}")
print(f"🃏 Decks:  {decks_df.shape}")
print(f"🤖 Models loaded from: {MODELS_PATH.absolute()}")

✅ Data and models loaded!
📦 Cards:  (98, 16)
🃏 Decks:  (480, 18)
🤖 Models loaded from: e:\Clash Royale\notebooks\..\backend\ml\models


In [3]:
# ====================================================
# 🔢 CREATE DECK VECTORS (Encode decks as numbers)
# ====================================================

print("🔢 ENCODING DECKS AS VECTORS")
print("=" * 60)

# Extract card lists from each deck
deck_card_lists = []
for _, row in decks_df.iterrows():
    cards = [row[f'card_{i}'] for i in range(1, 9)]
    deck_card_lists.append(cards)

# Use MultiLabelBinarizer to convert decks to binary vectors
# Each column = one card, value = 1 if present, 0 if not
mlb = MultiLabelBinarizer()
deck_vectors = mlb.fit_transform(deck_card_lists)

print(f"✅ Created deck vectors!")
print(f"📊 Shape: {deck_vectors.shape}")
print(f"   • {deck_vectors.shape[0]} decks")
print(f"   • {deck_vectors.shape[1]} unique cards as features")
print(f"\n🔍 Sample vector (first deck, first 20 features):")
print(deck_vectors[0][:20])

🔢 ENCODING DECKS AS VECTORS
✅ Created deck vectors!
📊 Shape: (480, 90)
   • 480 decks
   • 90 unique cards as features

🔍 Sample vector (first deck, first 20 features):
[0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]


In [4]:
# ====================================================
# 🔍 ENGINE 1: SIMILAR DECK FINDER (KNN)
# ====================================================

print("🔍 BUILDING SIMILAR DECK FINDER")
print("=" * 60)

# Train KNN model
knn_model = NearestNeighbors(
    n_neighbors=6,           # Find 6 similar (1 = itself, 5 = recommendations)
    metric='cosine',         # Cosine similarity (best for sparse data)
    algorithm='brute'
)

knn_model.fit(deck_vectors)

print("✅ KNN Model trained!")
print(f"📊 Neighbors per search: 5 similar decks")

# Save model and binarizer
joblib.dump(knn_model, MODELS_PATH / "similar_deck_finder.pkl")
joblib.dump(mlb, MODELS_PATH / "card_binarizer.pkl")

print(f"\n💾 Saved: similar_deck_finder.pkl")
print(f"💾 Saved: card_binarizer.pkl")

🔍 BUILDING SIMILAR DECK FINDER
✅ KNN Model trained!
📊 Neighbors per search: 5 similar decks

💾 Saved: similar_deck_finder.pkl
💾 Saved: card_binarizer.pkl


In [5]:
# ====================================================
# 🛠️ FUNCTION: FIND SIMILAR DECKS
# ====================================================

def find_similar_decks(user_deck, top_n=5):
    """
    Find decks most similar to the user's deck.
    
    Args:
        user_deck (list): List of 8 card names
        top_n (int): Number of similar decks to return
    
    Returns:
        DataFrame with similar decks and similarity scores
    """
    # Convert user deck to vector
    user_vector = mlb.transform([user_deck])
    
    # Find nearest neighbors
    distances, indices = knn_model.kneighbors(user_vector, n_neighbors=top_n + 1)
    
    # Get similar decks (skip first = itself if exists)
    similar_decks = []
    for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        if i == 0 and dist == 0:
            continue  # Skip if exact match (itself)
        if len(similar_decks) >= top_n:
            break
        
        deck = decks_df.iloc[idx]
        similarity = round((1 - dist) * 100, 2)
        
        similar_decks.append({
            'deck_id': deck['deck_id'],
            'cards': [deck[f'card_{i}'] for i in range(1, 9)],
            'archetype': deck['archetype'],
            'avg_elixir': deck['avg_elixir'],
            'win_rate': deck['win_rate'],
            'quality': deck['quality'],
            'similarity_score': similarity
        })
    
    return pd.DataFrame(similar_decks)


print("✅ Function `find_similar_decks()` created!")

✅ Function `find_similar_decks()` created!


In [6]:
# ====================================================
# 🧪 TEST: FIND SIMILAR DECKS
# ====================================================

print("🧪 TESTING SIMILAR DECK FINDER")
print("=" * 60)

# Sample user deck: Classic Hog Cycle
user_deck = ['Hog Rider', 'Musketeer', 'Ice Spirit', 'Skeletons',
             'Fireball', 'The Log', 'Cannon', 'Ice Golem']

print(f"🎴 USER DECK:")
for i, card in enumerate(user_deck, 1):
    print(f"   {i}. {card}")

print(f"\n🔍 FINDING TOP 5 SIMILAR DECKS...\n")

similar = find_similar_decks(user_deck, top_n=5)

for idx, row in similar.iterrows():
    print(f"{'─' * 60}")
    print(f"#{idx + 1} Similar Deck (ID: {row['deck_id']})")
    print(f"   🎯 Similarity:  {row['similarity_score']}%")
    print(f"   🏆 Archetype:   {row['archetype']}")
    print(f"   💧 Avg Elixir:  {row['avg_elixir']}")
    print(f"   📈 Win Rate:    {row['win_rate']}%")
    print(f"   ⭐ Quality:     {row['quality']}")
    print(f"   🃏 Cards:       {', '.join(row['cards'])}")

🧪 TESTING SIMILAR DECK FINDER
🎴 USER DECK:
   1. Hog Rider
   2. Musketeer
   3. Ice Spirit
   4. Skeletons
   5. Fireball
   6. The Log
   7. Cannon
   8. Ice Golem

🔍 FINDING TOP 5 SIMILAR DECKS...

────────────────────────────────────────────────────────────
#1 Similar Deck (ID: 238)
   🎯 Similarity:  100.0%
   🏆 Archetype:   Cycle
   💧 Avg Elixir:  2.62
   📈 Win Rate:    57.93%
   ⭐ Quality:     Good
   🃏 Cards:       Hog Rider, Musketeer, Ice Spirit, Skeletons, Fireball, The Log, Cannon, Ice Golem
────────────────────────────────────────────────────────────
#2 Similar Deck (ID: 2)
   🎯 Similarity:  100.0%
   🏆 Archetype:   Cycle
   💧 Avg Elixir:  2.62
   📈 Win Rate:    57.64%
   ⭐ Quality:     Good
   🃏 Cards:       Hog Rider, Musketeer, Ice Spirit, Skeletons, Fireball, The Log, Cannon, Ice Golem
────────────────────────────────────────────────────────────
#3 Similar Deck (ID: 236)
   🎯 Similarity:  100.0%
   🏆 Archetype:   Cycle
   💧 Avg Elixir:  2.62
   📈 Win Rate:    56.98%
   

In [7]:
# ====================================================
# 🔄 ENGINE 2: CARD REPLACEMENT SUGGESTER
# ====================================================

print("🔄 BUILDING CARD REPLACEMENT SUGGESTER")
print("=" * 60)

def suggest_card_replacements(user_deck, top_n=3):
    """
    Suggest better cards to replace the weakest card in the deck.
    
    Logic:
        1. Find weakest card (lowest win_rate)
        2. Suggest replacement cards with:
           - Similar elixir cost (±1)
           - Same type (Troop/Spell/Building)
           - Higher win rate
    
    Args:
        user_deck (list): User's 8 cards
        top_n (int): Number of suggestions per slot
    
    Returns:
        Dictionary with weakest card + replacement suggestions
    """
    # Find weakest card in deck
    deck_card_info = []
    for card in user_deck:
        if card in card_lookup:
            info = card_lookup[card]
            deck_card_info.append({
                'name': card,
                'elixir': info['elixir_cost'],
                'type': info['type'],
                'win_rate': info['win_rate']
            })
    
    deck_info_df = pd.DataFrame(deck_card_info)
    weakest = deck_info_df.loc[deck_info_df['win_rate'].idxmin()]
    
    # Find replacement candidates
    candidates = cards_df[
        (cards_df['elixir_cost'].between(weakest['elixir'] - 1, weakest['elixir'] + 1)) &
        (cards_df['type'] == weakest['type']) &
        (cards_df['win_rate'] > weakest['win_rate']) &
        (~cards_df['name'].isin(user_deck))  # Don't suggest existing cards
    ].nlargest(top_n, 'win_rate')
    
    suggestions = candidates[['name', 'elixir_cost', 'rarity',
                              'type', 'win_rate', 'usage_rate']].to_dict('records')
    
    return {
        'weakest_card': weakest['name'],
        'weakest_win_rate': weakest['win_rate'],
        'replacements': suggestions
    }


print("✅ Function `suggest_card_replacements()` created!")

🔄 BUILDING CARD REPLACEMENT SUGGESTER
✅ Function `suggest_card_replacements()` created!


In [8]:
# ====================================================
# 🧪 TEST: CARD REPLACEMENT SUGGESTIONS
# ====================================================

print("🧪 TESTING CARD REPLACEMENT SUGGESTER")
print("=" * 60)

print(f"🎴 USER DECK:")
for i, card in enumerate(user_deck, 1):
    print(f"   {i}. {card}")

print(f"\n🔄 ANALYZING WEAKEST CARD...\n")

result = suggest_card_replacements(user_deck, top_n=3)

print(f"❌ WEAKEST CARD: {result['weakest_card']}")
print(f"   Win Rate: {result['weakest_win_rate']:.2f}%\n")

print(f"✨ SUGGESTED REPLACEMENTS:\n")
if result['replacements']:
    for i, replacement in enumerate(result['replacements'], 1):
        print(f"   #{i} {replacement['name']}")
        print(f"      💧 Elixir:     {replacement['elixir_cost']}")
        print(f"      ⭐ Rarity:     {replacement['rarity']}")
        print(f"      📈 Win Rate:   {replacement['win_rate']:.2f}%")
        print(f"      🔥 Usage:      {replacement['usage_rate']:.2f}%\n")
else:
    print("   No suitable replacements found.")

🧪 TESTING CARD REPLACEMENT SUGGESTER
🎴 USER DECK:
   1. Hog Rider
   2. Musketeer
   3. Ice Spirit
   4. Skeletons
   5. Fireball
   6. The Log
   7. Cannon
   8. Ice Golem

🔄 ANALYZING WEAKEST CARD...

❌ WEAKEST CARD: The Log
   Win Rate: 52.04%

✨ SUGGESTED REPLACEMENTS:

   #1 Zap
      💧 Elixir:     2
      ⭐ Rarity:     Common
      📈 Win Rate:   57.45%
      🔥 Usage:      17.40%



In [9]:
# ====================================================
# 💪 ENGINE 3: COMPLETE DECK ANALYZER
# ====================================================

def analyze_deck(user_deck):
    """
    Complete deck analysis: strengths, weaknesses, suggestions.
    
    Returns:
        Dictionary with full analysis report
    """
    # Get card info
    deck_info = []
    for card in user_deck:
        if card in card_lookup:
            info = card_lookup[card]
            deck_info.append({
                'name': card,
                'elixir': info['elixir_cost'],
                'type': info['type'],
                'rarity': info['rarity'],
                'archetype': info['archetype'],
                'win_rate': info['win_rate'],
                'damage': info['damage'],
                'hp': info['hitpoints']
            })
    
    df = pd.DataFrame(deck_info)
    
    # Calculate metrics
    avg_elixir = df['elixir'].mean()
    num_troops = (df['type'] == 'Troop').sum()
    num_spells = (df['type'] == 'Spell').sum()
    num_buildings = (df['type'] == 'Building').sum()
    num_legendary = (df['rarity'] == 'Legendary').sum()
    avg_win_rate = df['win_rate'].mean()
    
    # Identify strengths
    strengths = []
    if avg_elixir <= 3.5:
        strengths.append("⚡ Fast cycle potential (low elixir)")
    if avg_elixir >= 4.5:
        strengths.append("💪 Strong push potential (high damage)")
    if num_spells >= 2:
        strengths.append("🎯 Good spell coverage")
    if num_buildings >= 1:
        strengths.append("🛡️ Defensive structure available")
    if num_troops >= 5:
        strengths.append("⚔️ Strong troop presence")
    if avg_win_rate >= 52:
        strengths.append("🏆 Cards have high win rates")
    
    # Identify weaknesses
    weaknesses = []
    if avg_elixir > 4.5:
        weaknesses.append("🐢 Slow cycle (high elixir)")
    if avg_elixir < 3.0:
        weaknesses.append("💔 Low damage potential")
    if num_spells == 0:
        weaknesses.append("❌ No spells - vulnerable to swarms")
    if num_buildings == 0 and 'Hog Rider' not in user_deck and 'Royal Giant' not in user_deck:
        weaknesses.append("🏚️ No defensive building")
    if num_troops < 4:
        weaknesses.append("⚠️ Few troops")
    if avg_win_rate < 48:
        weaknesses.append("📉 Cards have low win rates")
    
    # Most common archetype
    main_archetype = df['archetype'].mode()[0] if len(df['archetype']) > 0 else "Mixed"
    
    return {
        'avg_elixir': round(avg_elixir, 2),
        'avg_card_win_rate': round(avg_win_rate, 2),
        'composition': {
            'troops': int(num_troops),
            'spells': int(num_spells),
            'buildings': int(num_buildings),
            'legendary_count': int(num_legendary)
        },
        'primary_archetype': main_archetype,
        'strengths': strengths if strengths else ["⚖️ Balanced deck"],
        'weaknesses': weaknesses if weaknesses else ["✅ No major weaknesses"]
    }


print("✅ Function `analyze_deck()` created!")

✅ Function `analyze_deck()` created!


In [10]:
# ====================================================
# 🧪 TEST: COMPLETE DECK ANALYSIS
# ====================================================

print("🧪 COMPLETE DECK ANALYSIS")
print("=" * 60)

analysis = analyze_deck(user_deck)

print(f"🎴 USER DECK:")
for i, card in enumerate(user_deck, 1):
    print(f"   {i}. {card}")

print(f"\n📊 DECK METRICS:")
print(f"   💧 Average Elixir:        {analysis['avg_elixir']}")
print(f"   📈 Avg Card Win Rate:     {analysis['avg_card_win_rate']}%")
print(f"   🏆 Primary Archetype:     {analysis['primary_archetype']}")

print(f"\n🃏 DECK COMPOSITION:")
print(f"   ⚔️  Troops:     {analysis['composition']['troops']}")
print(f"   ✨ Spells:      {analysis['composition']['spells']}")
print(f"   🏰 Buildings:   {analysis['composition']['buildings']}")
print(f"   👑 Legendary:   {analysis['composition']['legendary_count']}")

print(f"\n💪 STRENGTHS:")
for s in analysis['strengths']:
    print(f"   {s}")

print(f"\n⚠️  WEAKNESSES:")
for w in analysis['weaknesses']:
    print(f"   {w}")

🧪 COMPLETE DECK ANALYSIS
🎴 USER DECK:
   1. Hog Rider
   2. Musketeer
   3. Ice Spirit
   4. Skeletons
   5. Fireball
   6. The Log
   7. Cannon
   8. Ice Golem

📊 DECK METRICS:
   💧 Average Elixir:        2.62
   📈 Avg Card Win Rate:     54.1%
   🏆 Primary Archetype:     Control

🃏 DECK COMPOSITION:
   ⚔️  Troops:     5
   ✨ Spells:      2
   🏰 Buildings:   1
   👑 Legendary:   1

💪 STRENGTHS:
   ⚡ Fast cycle potential (low elixir)
   🎯 Good spell coverage
   🛡️ Defensive structure available
   ⚔️ Strong troop presence
   🏆 Cards have high win rates

⚠️  WEAKNESSES:
   💔 Low damage potential


In [11]:
# ====================================================
# 🎯 ULTIMATE: COMPLETE DECK ANALYSIS & RECOMMENDATIONS
# ====================================================

def get_full_recommendation(user_deck):
    """
    Generate complete deck analysis with all recommendations.
    
    Returns:
        Full report combining all 3 engines
    """
    # 1. Deck analysis
    analysis = analyze_deck(user_deck)
    
    # 2. Similar decks
    similar = find_similar_decks(user_deck, top_n=3)
    
    # 3. Card replacements
    replacements = suggest_card_replacements(user_deck, top_n=3)
    
    return {
        'user_deck': user_deck,
        'analysis': analysis,
        'similar_decks': similar.to_dict('records'),
        'card_replacements': replacements
    }


print("=" * 70)
print("🎯 ULTIMATE DECK RECOMMENDATION REPORT")
print("=" * 70)

report = get_full_recommendation(user_deck)

print(f"\n🎴 YOUR DECK:")
for i, card in enumerate(report['user_deck'], 1):
    print(f"   {i}. {card}")

print(f"\n{'─' * 60}")
print(f"📊 ANALYSIS")
print(f"{'─' * 60}")
print(f"   🏆 Archetype:    {report['analysis']['primary_archetype']}")
print(f"   💧 Avg Elixir:   {report['analysis']['avg_elixir']}")
print(f"   📈 Avg WinRate:  {report['analysis']['avg_card_win_rate']}%")

print(f"\n💪 STRENGTHS:")
for s in report['analysis']['strengths']:
    print(f"   {s}")

print(f"\n⚠️  WEAKNESSES:")
for w in report['analysis']['weaknesses']:
    print(f"   {w}")

print(f"\n{'─' * 60}")
print(f"🔄 SUGGESTED IMPROVEMENT")
print(f"{'─' * 60}")
print(f"   ❌ Weakest Card:  {report['card_replacements']['weakest_card']}")
print(f"   ✨ Top 3 Replacements:")
for i, r in enumerate(report['card_replacements']['replacements'], 1):
    print(f"      {i}. {r['name']} (Win: {r['win_rate']:.2f}%)")

print(f"\n{'─' * 60}")
print(f"🔍 TOP 3 SIMILAR DECKS")
print(f"{'─' * 60}")
for i, deck in enumerate(report['similar_decks'][:3], 1):
    print(f"\n   #{i} Similarity: {deck['similarity_score']}% | Win Rate: {deck['win_rate']}%")
    print(f"      {', '.join(deck['cards'][:4])}")
    print(f"      {', '.join(deck['cards'][4:])}")

print(f"\n{'=' * 70}")

🎯 ULTIMATE DECK RECOMMENDATION REPORT

🎴 YOUR DECK:
   1. Hog Rider
   2. Musketeer
   3. Ice Spirit
   4. Skeletons
   5. Fireball
   6. The Log
   7. Cannon
   8. Ice Golem

────────────────────────────────────────────────────────────
📊 ANALYSIS
────────────────────────────────────────────────────────────
   🏆 Archetype:    Control
   💧 Avg Elixir:   2.62
   📈 Avg WinRate:  54.1%

💪 STRENGTHS:
   ⚡ Fast cycle potential (low elixir)
   🎯 Good spell coverage
   🛡️ Defensive structure available
   ⚔️ Strong troop presence
   🏆 Cards have high win rates

⚠️  WEAKNESSES:
   💔 Low damage potential

────────────────────────────────────────────────────────────
🔄 SUGGESTED IMPROVEMENT
────────────────────────────────────────────────────────────
   ❌ Weakest Card:  The Log
   ✨ Top 3 Replacements:
      1. Zap (Win: 57.45%)

────────────────────────────────────────────────────────────
🔍 TOP 3 SIMILAR DECKS
────────────────────────────────────────────────────────────

   #1 Similarity: 100.0% |

In [12]:
# ====================================================
# 💾 SAVE METADATA FOR BACKEND USE
# ====================================================

import pickle

# Save card lookup for backend
with open(MODELS_PATH / "card_lookup.pkl", "wb") as f:
    pickle.dump(card_lookup, f)

print("💾 Saved: card_lookup.pkl")

# List all saved models
print("\n📁 ALL SAVED ML FILES:")
for file in sorted(MODELS_PATH.glob("*.pkl")):
    size_kb = file.stat().st_size / 1024
    print(f"   ✅ {file.name:35} ({size_kb:.1f} KB)")

💾 Saved: card_lookup.pkl

📁 ALL SAVED ML FILES:
   ✅ archetype_classifier.pkl            (961.3 KB)
   ✅ card_binarizer.pkl                  (2.0 KB)
   ✅ card_lookup.pkl                     (11.7 KB)
   ✅ deck_strength_scorer.pkl            (705.0 KB)
   ✅ label_encoder.pkl                   (0.6 KB)
   ✅ scaler.pkl                          (1.4 KB)
   ✅ similar_deck_finder.pkl             (169.3 KB)
   ✅ win_rate_predictor.pkl              (2150.9 KB)


In [13]:
# ====================================================
# 📝 RECOMMENDATION ENGINE SUMMARY
# ====================================================

print("=" * 70)
print("🎉 RECOMMENDATION ENGINE COMPLETE!")
print("=" * 70)

print(f"""
🔍 ENGINE 1: SIMILAR DECK FINDER
   ✅ Algorithm:  KNN + Cosine Similarity
   ✅ Function:   find_similar_decks(user_deck, top_n=5)
   ✅ Saved:      similar_deck_finder.pkl

🔄 ENGINE 2: CARD REPLACEMENT SUGGESTER
   ✅ Algorithm:  Rule-based with win rate analysis
   ✅ Function:   suggest_card_replacements(user_deck)

💪 ENGINE 3: DECK ANALYZER
   ✅ Function:   analyze_deck(user_deck)
   ✅ Provides:   Strengths, weaknesses, composition

🎯 ULTIMATE FUNCTION
   ✅ Function:   get_full_recommendation(user_deck)
   ✅ Returns:    Complete report with all engines

📂 ML MODELS COMPLETE:
   ✅ win_rate_predictor.pkl
   ✅ deck_strength_scorer.pkl
   ✅ archetype_classifier.pkl
   ✅ similar_deck_finder.pkl
   ✅ card_binarizer.pkl
   ✅ scaler.pkl
   ✅ label_encoder.pkl
   ✅ card_lookup.pkl

🚀 ML PART = 100% COMPLETE!
🚀 NEXT: BUILD FLASK BACKEND API!
""")

🎉 RECOMMENDATION ENGINE COMPLETE!

🔍 ENGINE 1: SIMILAR DECK FINDER
   ✅ Algorithm:  KNN + Cosine Similarity
   ✅ Function:   find_similar_decks(user_deck, top_n=5)
   ✅ Saved:      similar_deck_finder.pkl

🔄 ENGINE 2: CARD REPLACEMENT SUGGESTER
   ✅ Algorithm:  Rule-based with win rate analysis
   ✅ Function:   suggest_card_replacements(user_deck)

💪 ENGINE 3: DECK ANALYZER
   ✅ Function:   analyze_deck(user_deck)
   ✅ Provides:   Strengths, weaknesses, composition

🎯 ULTIMATE FUNCTION
   ✅ Function:   get_full_recommendation(user_deck)
   ✅ Returns:    Complete report with all engines

📂 ML MODELS COMPLETE:
   ✅ win_rate_predictor.pkl
   ✅ deck_strength_scorer.pkl
   ✅ archetype_classifier.pkl
   ✅ similar_deck_finder.pkl
   ✅ card_binarizer.pkl
   ✅ scaler.pkl
   ✅ label_encoder.pkl
   ✅ card_lookup.pkl

🚀 ML PART = 100% COMPLETE!
🚀 NEXT: BUILD FLASK BACKEND API!

